In [1]:
from google.colab import drive
import os
import torch
import numpy as np

drive.mount('/content/drive')

project_path = '/content/drive/MyDrive/Colab_Notebooks'
os.chdir(project_path)

!pip install -r requirements.txt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
class Two_layer_classifier:
    def __init__(self, logistic_model, ensemble_models, threshold=0.8, device="cuda"):
        self.logistic = logistic_model
        self.ensemble_models = ensemble_models
        self.threshold = threshold
        self.device = device

    def predict(self, text):
        logistic_pred, logistic_conf = self._logistic_predict(text)

        if logistic_conf >= self.threshold:
            return logistic_pred
        else:
            return self._ensemble_predict(text)

    def _logistic_predict(self, text):
        model, vectorizer = self.logistic
        X_test_vectorized = vectorizer.transform_phrase(text)
        y_pred = model.model.predict(X_test_vectorized)
        y_probs = model.model.predict_proba(X_test_vectorized)
        confidence_scores = y_probs.max(axis=1)
        print(f"[Logistic] Pred: {y_pred[0]}, Confidence: {confidence_scores[0]}")
        return y_pred[0], confidence_scores[0]

    def _ensemble_predict(self, text):
        all_preds = []
        for model, tokenizer in self.ensemble_models:
            inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
            inputs = {k: v.to(self.device) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = model(**inputs)

            pred = torch.argmax(outputs.logits, dim=1).cpu().item()
            all_preds.append(pred)

        return np.bincount(all_preds).argmax()



In [3]:
from model_utils import CustomClassifier, load_model_pickle

model_roberta, tok_roberta = load_model_pickle("Roberta/best_model","roberta-base")
model_distil, tok_distil = load_model_pickle("DistilRoberta/best_model","distilroberta-base")

ensemble_models = [
    (model_roberta.to("cuda"), tok_roberta),
    (model_distil.to("cuda"), tok_distil),
]


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
import tfidf_logistic_regression_randomsearch as tfidf
import os

vectorizer = tfidf.FullTextTfidfVectorizer()
vectorizer.load(os.path.join("logistic/", "vectorizer_binary.joblib"))

model = tfidf.TfidfLogisticModel()
model.load(os.path.join("logistic/", "logistic_binary.joblib"))

logistic = model, vectorizer

/usr/local/lib/python3.11/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.2.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.2.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.2.2 when using version 1.6.1. This might lead to breaking c

In [11]:
pipeline = Two_layer_classifier(
    logistic_model= logistic,
    ensemble_models= ensemble_models,
    threshold=0.8
    )

print("predicted label: ",pipeline.predict("you fucking faggot"))

[Logistic] Pred: 1, Confidence: 0.6571108351376422
daje
etichetta predetta:  1


In [6]:
import gradio as gr

def fun(text):
    if not text.strip():
        return "⛔ Please enter some text to analyze"

    try:
        prediction = pipeline.predict(text)
        print(prediction)

        if isinstance(prediction, np.ndarray):
            prediction = int(prediction.item())
        elif hasattr(prediction, 'item'):
            prediction = prediction.item()

        label_mapping = {
            0: "✅ Not hate speech",
            1: "⚠️ Hate speech detected",
        }
        return label_mapping.get(int(prediction), "❓ Unknown result")

    except Exception as e:
        return f"🔧 Error during prediction: {str(e)}"

with gr.Blocks(theme=gr.themes.Soft(), title="Hate Speech Detector") as demo:
    gr.Markdown("""
    # 🛡️ Hate Speech Detector
    **Real-time analysis of text content**
    """)

    with gr.Row():
        input_text = gr.Textbox(
            lines=4,
            placeholder="Enter your text here...",
            label="Input Text",
            elem_id="input-box"
        )

        output_label = gr.Label(
            label="Analysis Result",
            elem_id="result-box",
            value="Awaiting input..."
        )

    with gr.Row():
        analyze_btn = gr.Button("Analyze", variant="primary")
        clear_btn = gr.Button("Clear")

    analyze_btn.click(
        fn=fun,
        inputs=input_text,
        outputs=output_label
    )

    clear_btn.click(
        fn=lambda: ("", "Awaiting input..."),
        inputs=None,
        outputs=[input_text, output_label]
    )

demo.launch()

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0f0c3fecba519b3750.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
